# 03 — Scaffold a New Agent

This notebook walks you through creating a new agent using the scaffolding script,
verifying the generated files, running the generated tests, and deploying the agent.

## Prerequisites

- Python 3.11+ with [uv](https://docs.astral.sh/uv/) installed
- Dependencies installed (`uv sync --group dev`)

## Configuration

Set the name for your new agent below. Must be kebab-case (e.g., `my-agent`).

In [ ]:
# Change this to your desired agent name (kebab-case)
AGENT_NAME = "my-demo-agent"
MODEL = "gpt-4o"

## Step 1: Scaffold the Agent

Run the scaffolding script with `--name` to generate all agent files, test stubs, and registry entry.

In [ ]:
!python ../scripts/create_agent.py --name {AGENT_NAME} --model {MODEL}

## Step 2: Verify Generated Files

Check that all expected files were created in the agent and test directories.

In [ ]:
import os

module_name = AGENT_NAME.replace("-", "_")
agent_dir = f"../agents/{module_name}"
test_dir = f"../tests/{module_name}"

print(f"Agent directory: {agent_dir}/")
for root, dirs, files in os.walk(agent_dir):
    level = root.replace(agent_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for f in files:
        print(f"{sub_indent}{f}")

print(f"\nTest directory: {test_dir}/")
for root, dirs, files in os.walk(test_dir):
    level = root.replace(test_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for f in files:
        print(f"{sub_indent}{f}")

## Step 3: Inspect Generated Config

View the generated configuration to confirm the agent name, model, and paths.

In [ ]:
config_path = f"../agents/{module_name}/config.py"
print(open(config_path).read())

## Step 4: Run Generated Tests

Run the unit tests generated for your new agent to verify the scaffolding is correct.

In [ ]:
!python -m pytest ../tests/{module_name}/ -v -m "not integration"

## Step 5: Deploy the Agent

Deploy your scaffolded agent to Azure AI Foundry.

### Integration Toggles (Optional)

Each agent controls its own tools via its `config.py`. Use the toggles below to override at runtime for quick testing. To persist changes, update the agent's `config.py` directly.

In [ ]:
# Toggle integrations — set to True/False as needed
CODE_INTERPRETER_ENABLED = False
WEB_SEARCH_ENABLED = False
GITHUB_ENABLED = False
GITHUB_PROJECT_CONNECTION_ID = ""  # Required when GITHUB_ENABLED is True

In [ ]:
from agents._base.client import reset_client
from agents.registry import REGISTRY

# Reset singleton to ensure we use the current .env endpoint
reset_client()

# Deploy the agent via the factory with integration overrides
entry = REGISTRY.get_agent(AGENT_NAME)
config = entry.config_class()

# Apply integration toggle overrides if the toggle cell was run;
# otherwise fall back to the agent's config.py defaults.
if "CODE_INTERPRETER_ENABLED" in dir():
    config.code_interpreter_enabled = CODE_INTERPRETER_ENABLED
if "WEB_SEARCH_ENABLED" in dir():
    config.web_search_enabled = WEB_SEARCH_ENABLED
if "GITHUB_ENABLED" in dir():
    config.github_enabled = GITHUB_ENABLED
if "GITHUB_PROJECT_CONNECTION_ID" in dir() and GITHUB_PROJECT_CONNECTION_ID:
    config.github_project_connection_id = GITHUB_PROJECT_CONNECTION_ID

print(f"Deploying agent: {AGENT_NAME}")
print(f"  Code Interpreter: {config.code_interpreter_enabled}")
print(f"  Web Search:       {config.web_search_enabled}")
print(f"  GitHub MCP:       {config.github_enabled}")

agent = entry.factory(config)
print(f"\n✓ Agent '{agent.name}' deployed successfully (id: {agent.id})")

## Step 6: Delete the Agent (Optional)

Remove the scaffolded agent from the codebase — deletes the agent directory, test directory, registry entry, and pytest marker.

⚠️ **This is destructive and cannot be undone.** Only run this if you want to completely remove the agent.

In [ ]:
# Uncomment to delete the scaffolded agent (this is irreversible)
#!python ../scripts/delete_agent.py --name {AGENT_NAME} --force

## Next Steps

1. Edit `agents/{module_name}/instructions.md` with your agent's system prompt
2. Add custom tools in `agents/{module_name}/tools/`
3. (Optional) Set default integrations in `agents/{module_name}/config.py` — toggle `code_interpreter_enabled`, `web_search_enabled`, or `github_enabled`
4. Use **02_build_and_run_agent.ipynb** to deploy and test interactively
5. See the [Scaffolding Guide](../docs/scaffolding-guide.md) for full reference